In [1]:
import os

cache_root = "/scratch/gilbreth/abelde"

os.environ["HOME"]              = cache_root
os.environ["PIP_CACHE_DIR"]     = os.path.join(cache_root, ".cache", "pip")
os.environ["TORCH_HOME"]        = os.path.join(cache_root, ".cache", "torch")
os.environ["HF_HOME"]           = os.path.join(cache_root, ".cache", "huggingface")
os.environ["XDG_CACHE_HOME"]    = os.path.join(cache_root, ".cache")
os.environ["TMPDIR"]            = os.path.join(cache_root, "tmp")

os.makedirs(os.environ["PIP_CACHE_DIR"], exist_ok=True)
os.makedirs(os.environ["TORCH_HOME"], exist_ok=True)
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(os.environ["XDG_CACHE_HOME"], exist_ok=True)
os.makedirs(os.environ["TMPDIR"], exist_ok=True)

print("Cache directories set under:", cache_root)

Cache directories set under: /scratch/gilbreth/abelde


In [3]:
import importlib
import sys
import os
import torch

# ──────────── Paths ────────────
repo_root = "/scratch/gilbreth/abelde/Vision_Graphics/3DGS/gaussian-splatting"
data_path = "/scratch/gilbreth/abelde/Vision_Graphics/3DGS/datasets/garden"
output_dir = os.path.join(repo_root, "output/debug_garden")

# ──────────── Fix LD_LIBRARY_PATH for CUDA extensions ────────────
torch_lib = os.path.join(os.path.dirname(torch.__file__), "lib")
os.environ["LD_LIBRARY_PATH"] = torch_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# ──────────── Build CLI args ────────────
sys.argv = [
    "train.py",
    "--source_path", data_path,
    "--model_path", output_dir,
    "--iterations", "1000",          # short run for debugging
    "--test_iterations", "500", "1000",
    "--save_iterations", "1000",
    "--eval",
    "--disable_viewer",              # no GUI server needed
]

# ──────────── Import and run ────────────
import train
importlib.reload(train)

from arguments import ModelParams, PipelineParams, OptimizationParams
from argparse import ArgumentParser
from utils.general_utils import safe_state
from gaussian_renderer import network_gui

parser = ArgumentParser(description="Training script parameters")
lp = ModelParams(parser)
op = OptimizationParams(parser)
pp = PipelineParams(parser)
parser.add_argument('--ip', type=str, default="127.0.0.1")
parser.add_argument('--port', type=int, default=6009)
parser.add_argument('--debug_from', type=int, default=-1)
parser.add_argument('--detect_anomaly', action='store_true', default=False)
parser.add_argument('--test_iterations', nargs='+', type=int, default=[7_000, 30_000])
parser.add_argument('--save_iterations', nargs='+', type=int, default=[7_000, 30_000])
parser.add_argument('--quiet', action='store_true')
parser.add_argument('--disable_viewer', action='store_true', default=False)
parser.add_argument('--checkpoint_iterations', nargs='+', type=int, default=[])
parser.add_argument('--start_checkpoint', type=str, default=None)

args = parser.parse_args(sys.argv[1:])
args.save_iterations.append(args.iterations)

print("Optimizing " + args.model_path)

safe_state(args.quiet)
torch.autograd.set_detect_anomaly(args.detect_anomaly)

# ──────────── Launch training (set breakpoints inside train.training) ────────────
train.training(
    lp.extract(args), op.extract(args), pp.extract(args),
    args.test_iterations, args.save_iterations,
    args.checkpoint_iterations, args.start_checkpoint, args.debug_from
)

Optimizing /scratch/gilbreth/abelde/Vision_Graphics/3DGS/gaussian-splatting/output/debug_garden
Output folder: /scratch/gilbreth/abelde/Vision_Graphics/3DGS/gaussian-splatting/output/debug_garden [09/03 23:21:57]
------------LLFF HOLD------------- [09/03 23:28:41]
Reading camera 185/185 [09/03 23:28:41]
Loading Training Cameras [09/03 23:30:23]
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly specify '--resolution/-r' as 1 [09/03 23:30:41]


KeyboardInterrupt: 

In [1]:
print("Training completed successfully.")

Training completed successfully.
